In [1]:
!pip install langchain-groq langchain langchain_core TimezoneFinder

In [2]:
import os, sys
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
GEOAPIFY_API_KEY = userdata.get("GEOAPIFY_API_KEY")
OPENAQ_API_KEY   = userdata.get("OPENAQ_API_KEY")
FIRMS_MAP_KEY    = userdata.get("FIRMS_MAP_KEY")

def local_now_iso(tz_name: str = "UTC") -> str:
    try:
        return datetime.now(ZoneInfo(tz_name)).isoformat()
    except Exception:
        return datetime.now(ZoneInfo("UTC")).isoformat()

In [ ]:
import os
import json
import hashlib
from pathlib import Path
from diskcache import Cache

api_cache = Cache("./cache")
FIXTURES_DIR = Path("./fixtures")
FIXTURES_DIR.mkdir(exist_ok=True)

# Toggle this to True before the demo
DEMO_MODE = os.getenv("DEMO_MODE", "False").lower() == "true"

def get_cached_or_live(cache_key: str, fetch_func, ttl_seconds: int = 3600, *args, **kwargs):
    """Checks the fixtures folder (if in demo mode), then the cache, and finally the live API."""
    
    # 1. Demo Mode: Load from JSON file
    if DEMO_MODE:
        fixture_path = FIXTURES_DIR / f"{cache_key}.json"
        if fixture_path.exists():
            with open(fixture_path, "r") as f:
                return json.load(f)
        print(f"Warning: Demo fixture {fixture_path} missing. Falling back to cache/live.")

    # Generate a unique hash for the coordinates/args
    arg_hash = hashlib.md5(f"{args}_{kwargs}".encode()).hexdigest()
    full_key = f"{cache_key}_{arg_hash}"

    # 2. Cache Hit: Load from diskcache
    if full_key in api_cache:
        return api_cache[full_key]

    # 3. Live API Fetch
    live_data = fetch_func(*args, **kwargs)
    
    # Save to cache to prevent rate-limiting during testing
    api_cache.set(full_key, live_data, expire=ttl_seconds)
    
    # Auto-save a fixture so you have it ready for demo mode
    with open(FIXTURES_DIR / f"{cache_key}.json", "w") as f:
        json.dump(live_data, f, indent=2)
        
    return live_data

In [12]:
import json
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from typing import Optional, TypedDict, List, Dict, Any
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_core.output_parsers import JsonOutputParser
from geopy.geocoders import Nominatim
from timezonefinder import TimezoneFinder

In [ ]:
class QueryInfo(BaseModel):
    city: Optional[str] = Field(default=None, description="The city or location to analyze.")
    country_or_continent: Optional[str] = Field(default=None, description="The continent or country of the city to avoid ambiguity.")
    start_date: Optional[str] = Field(default=None, description="The start date in YYYY-MM-DD format.")
    end_date: Optional[str] = Field(default=None, description="The end date in YYYY-MM-DD format.")
    start_time: Optional[str] = Field(default=None, description="The start time in HH:MM 24h format.")
    end_time: Optional[str] = Field(default=None, description="The end time in HH:MM 24h format.")
    latitude: Optional[float] = Field(default=None, description="Latitude of the specific point to analyze.")
    longitude: Optional[float] = Field(default=None, description="Longitude of the specific point to analyze.")
class InvestigationState(TypedDict):
    question: str
    location: Dict[str, Any]
    messages: List[Any]
    evidence: List[Dict[str, Any]]
    iterations: int
    max_iterations: int

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
parser = JsonOutputParser(pydantic_object=QueryInfo)

prompt = PromptTemplate(
    template=(
        "You are a precise parameter extractor for the Open-Meteo historical weather API.\n\n"
        "Extract these parameters from the user input and ensure they are included in the final output:\n"
        "- city (The city name)\n"
        "- country_or_continent (Infer the continent or country logically from the city to resolve duplicate names/ambiguity)\n"
        "- latitude (Look up or infer the exact decimal latitude coordinates for the city. MUST NOT BE NULL)\n"
        "- longitude (Look up or infer the exact decimal longitude coordinates for the city. MUST NOT BE NULL)\n"
        "- start_date (Set as 'TODAY', 'YESTERDAY', 'TOMORROW' or exact YYYY-MM-DD if mentioned)\n"
        "- end_date (Set as 'TODAY', 'YESTERDAY', 'TOMORROW' or exact YYYY-MM-DD if mentioned)\n"
        "- start_time (Set as 'NOW' or exact HH:MM in 24-hour format if mentioned)\n"
        "- end_time (Set as 'NOW' or exact HH:MM in 24-hour format if mentioned)\n\n"
        "Rules:\n"
        "- If 'now', 'current', 'currently', 'right now' are implied for date/time, use 'TODAY' and 'NOW'.\n"
        "- If only one date is provided, use it for both start_date & end_date.\n"
        "- If no date or time is specified, return 'TODAY' and 'NOW'.\n"
        "- Always provide precise numerical latitude and longitude for the location.\n"
        "- Never invent or guess dates/times.\n\n"
        "{format_instructions}\n"
        "User input: {user_input}\n"
    ),
    input_variables=["user_input"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

extraction_chain = prompt | llm | parser

def get_location_details(city_name: str, country_or_continent: str) -> tuple[ZoneInfo, Optional[float], Optional[float]]:
    """Reuses the geocode tool's underlying lookup so extraction and the agent's own
    geocode calls stay consistent."""
    result = geocode.invoke({"city": city_name, "country_or_continent": country_or_continent})
    if not result.get("error") and result.get("location"):
        lat, lon = result["location"]["lat"], result["location"]["lon"]
        tz_name = result["measurements"].get("tz_name", "UTC")
        return ZoneInfo(tz_name), lat, lon
    return ZoneInfo("UTC"), None, None


def data_extraction(user_input: str) -> str:
    raw_result = extraction_chain.invoke({"user_input": user_input})
    city_name = raw_result.get("city", "")
    location_context = raw_result.get("country_or_continent", "")
    tz, geo_lat, geo_lon = get_location_details(city_name, location_context)
    local_now = datetime.now(tz)
    final_lat = geo_lat if geo_lat is not None else raw_result.get("latitude")
    final_lon = geo_lon if geo_lon is not None else raw_result.get("longitude")
    date_mapping = {
        "TODAY": local_now.strftime("%Y-%m-%d"),
        "YESTERDAY": (local_now - timedelta(days=1)).strftime("%Y-%m-%d"),
        "TOMORROW": (local_now + timedelta(days=1)).strftime("%Y-%m-%d"),
    }
    start_date = date_mapping.get(raw_result.get("start_date"), raw_result.get("start_date"))
    end_date = date_mapping.get(raw_result.get("end_date"), raw_result.get("end_date"))
    start_time = local_now.strftime("%H:%M") if raw_result.get("start_time") == "NOW" else raw_result.get("start_time")
    end_time = local_now.strftime("%H:%M") if raw_result.get("end_time") == "NOW" else raw_result.get("end_time")
    result = QueryInfo(
        city=city_name,
        country_or_continent=location_context,
        start_date=start_date,
        end_date=end_date,
        start_time=start_time,
        end_time=end_time,
        latitude=final_lat,
        longitude=final_lon,
    )
    return result.model_dump_json(indent=2)

In [ ]:
def fetch_geo_live(lat: float, lon: float, radius_m: int, categories: str):
    """Raw Geoapify Places API call."""
    url = "https://api.geoapify.com/v2/places"
    params = {
        "categories": categories, 
        "filter": f"circle:{lon},{lat},{radius_m}", 
        "limit": 20, 
        "apiKey": GEOAPIFY_API_KEY
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return [{"name": f["properties"].get("name"), "category": f["properties"].get("categories")} for f in r.json().get("features", [])]

@tool
def geo_adapter(lat: float, lon: float, tz_name: str = "UTC", radius_m: int = 5000, categories: str = "populated_place") -> dict:
    """Find nearby geographic features / infrastructure around a point using Geoapify Places."""
    try:
        # Cache for 24 hours (86400 seconds) since infrastructure rarely changes
        features = get_cached_or_live("geoapify_places", fetch_geo_live, 86400, lat, lon, radius_m, categories)
        return {
            "source": "geoapify", 
            "timestamp": local_now_iso(tz_name), 
            "location": {"lat": lat, "lon": lon}, 
            "measurements": {"features": features}
        }
    except Exception as e:
        return {"source": "geoapify", "error": True, "raw_reference": str(e)}

In [ ]:
tf = TimezoneFinder()

def fetch_geocode_live(city: str, country_or_continent: str):
    """Raw Geoapify Geocoding API call."""
    url = "https://api.geoapify.com/v1/geocode/search"
    params = {"text": f"{city}, {country_or_continent}".strip(", "), "limit": 1, "apiKey": GEOAPIFY_API_KEY}
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    feats = r.json().get("features", [])
    if not feats:
        raise ValueError(f"No geocoding result for '{city}'")
    
    lon_val, lat_val = feats[0]["geometry"]["coordinates"]
    tz_val = tf.timezone_at(lng=lon_val, lat=lat_val) or "UTC"
    return {"lat": lat_val, "lon": lon_val, "matched_name": feats[0]["properties"].get("formatted"), "tz_name": tz_val}

@tool
def geocode(city: str, country_or_continent: str = "") -> dict:
    """Resolve a city name to latitude/longitude and IANA timezone using Geoapify Geocoding."""
    try:
        # Cache for 24 hours (86400 seconds)
        data = get_cached_or_live("geoapify_geocode", fetch_geocode_live, 86400, city, country_or_continent)
        return {
            "source": "geoapify_geocode", 
            "timestamp": local_now_iso(data["tz_name"]), 
            "location": {"lat": data["lat"], "lon": data["lon"]}, 
            "measurements": {"matched_name": data["matched_name"], "tz_name": data["tz_name"]}
        }
    except Exception as e:
        return {"source": "geoapify_geocode", "error": True, "raw_reference": str(e)}

In [ ]:
def fetch_weather_live(lat: float, lon: float, tz_name: str) -> dict:
    """Raw Open-Meteo API call."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "current": "temperature_2m,wind_speed_10m,wind_direction_10m,precipitation",
        "timezone": tz_name,
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json().get("current", {})

@tool
def weather_adapter(lat: float, lon: float, tz_name: str = "UTC") -> dict:
    """Get current weather (temperature, wind, precipitation) for a location via Open-Meteo, in local time."""
    try:
        # Pass the fetch function to the cache manager
        current = get_cached_or_live("open_meteo_weather", fetch_weather_live, 3600, lat, lon, tz_name)
        
        return {
            "source": "open-meteo",
            "timestamp": current.get("time", local_now_iso(tz_name)),
            "location": {"lat": lat, "lon": lon},
            "measurements": current,
        }
    except Exception as e:
        return {"source": "open-meteo", "error": True, "raw_reference": str(e)}

In [ ]:
@tool
def air_quality_adapter(lat: float, lon: float, tz_name: str = "UTC", radius_m: int = 25000) -> dict:
    """Get the nearest air-quality station's latest PM2.5/PM10/NO2/O3/CO/SO2 readings via OpenAQ."""
    headers = {"X-API-Key": OPENAQ_API_KEY}
    try:
        loc_r = requests.get(
            "https://api.openaq.org/v3/locations",
            params={"coordinates": f"{lat},{lon}", "radius": radius_m, "limit": 1},
            headers=headers, timeout=10,
        )
        loc_r.raise_for_status()
        results = loc_r.json().get("results", [])
        if not results:
            raise ValueError("No OpenAQ station within radius")
        loc_id = results[0]["id"]

        latest_r = requests.get(
            f"https://api.openaq.org/v3/locations/{loc_id}/latest",
            headers=headers, timeout=10,
        )
        latest_r.raise_for_status()
        measurements = {m["parameter"]["name"]: m["value"] for m in latest_r.json().get("results", [])}

        return {
            "source": "openaq",
            "timestamp": local_now_iso(tz_name),
            "location": {"lat": lat, "lon": lon},
            "measurements": measurements,
            "raw_reference": latest_r.url,
        }
    except Exception as e:
        return {"source": "openaq", "timestamp": local_now_iso(tz_name), "location": {"lat": lat, "lon": lon},
                "measurements": {}, "raw_reference": str(e), "error": True}

In [ ]:
import math
from typing import Tuple

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate great-circle distance in kilometers between two points."""
    R = 6371.0  # Earth radius in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))


def calculate_bearing(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate compass bearing (0-360 deg) from point 1 toward point 2."""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - (math.sin(lat1) * math.cos(lat2) * math.cos(dlon))
    initial_bearing = math.atan2(x, y)
    return (math.degrees(initial_bearing) + 360) % 360

def is_location_downwind(
    source_lat: float, 
    source_lon: float, 
    target_lat: float, 
    target_lon: float, 
    wind_direction_deg: float, 
    tolerance_deg: float = 45.0
) -> Tuple[bool, float, float]:
    """
    Checks if target is downwind of source given the meteorological wind direction.
    (wind_direction_deg: direction the wind is blowing FROM).
    Returns (is_downwind, distance_km, bearing_to_target).
    """
    dist_km = haversine_distance(source_lat, source_lon, target_lat, target_lon)
    bearing_source_to_target = calculate_bearing(source_lat, source_lon, target_lat, target_lon)
    
    # Wind blows towards the opposite direction (wind_from + 180)
    wind_vector_to = (wind_direction_deg + 180) % 360
    
    angle_diff = abs((bearing_source_to_target - wind_vector_to + 180) % 360 - 180)
    return (angle_diff <= tolerance_deg), dist_km, bearing_source_to_target

def fetch_firms_live(lat: float, lon: float, buffer_deg: float, days: int):
    """Raw NASA FIRMS API call with geospatial math applied to the rows."""
    bbox = f"{lon - buffer_deg},{lat - buffer_deg},{lon + buffer_deg},{lat + buffer_deg}"
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_MAP_KEY}/VIIRS_NOAA20_NRT/{bbox}/{days}"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    lines = r.text.strip().split("\n")
    hotspots = []
    
    if len(lines) > 1:
        # Skip the CSV header row
        for row in lines[1:]:
            cols = row.split(",")
            # NASA FIRMS CSV: latitude is index 0, longitude is index 1
            fire_lat, fire_lon = float(cols[0]), float(cols[1])
            
            # Use the deterministic math functions you added earlier
            dist = haversine_distance(fire_lat, fire_lon, lat, lon)
            bearing = calculate_bearing(fire_lat, fire_lon, lat, lon)
            
            hotspots.append({
                "fire_lat": fire_lat, 
                "fire_lon": fire_lon, 
                "distance_to_city_km": round(dist, 2), 
                "bearing_to_city_deg": round(bearing, 2)
            })
            
    # Sort by closest distance so the agent prioritizes the most relevant threats
    hotspots.sort(key=lambda x: x["distance_to_city_km"])
    return {"hotspot_count": len(hotspots), "closest_fires": hotspots[:5]}

@tool
def fire_adapter(lat: float, lon: float, tz_name: str = "UTC", buffer_deg: float = 1.0, days: int = 1) -> dict:
    """Get active fire/hotspot detections and calculate their distance/bearing to the target location."""
    try:
        # Cache for 2 hours (7200 seconds)
        measurements = get_cached_or_live("nasa_firms", fetch_firms_live, 7200, lat, lon, buffer_deg, days)
        return {
            "source": "nasa_firms", 
            "timestamp": local_now_iso(tz_name), 
            "location": {"lat": lat, "lon": lon}, 
            "measurements": measurements
        }
    except Exception as e:
        return {"source": "nasa_firms", "error": True, "raw_reference": str(e)}

In [ ]:
def fetch_earthquake_live(lat: float, lon: float, radius_km: int, days: int, min_mag: float):
    """Raw USGS Earthquake API call."""
    from datetime import timedelta, timezone as dt_timezone, datetime as dt
    end = dt.now(dt_timezone.utc)
    start = end - timedelta(days=days)
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    params = {
        "format": "geojson", 
        "latitude": lat, 
        "longitude": lon,
        "maxradiuskm": radius_km, 
        "starttime": start.date().isoformat(),
        "endtime": end.date().isoformat(), 
        "minmagnitude": min_mag,
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    features = r.json().get("features", [])
    
    # Extract just the relevant fields to save LLM tokens
    return [{"mag": f["properties"]["mag"], "place": f["properties"]["place"], "time": f["properties"]["time"]} for f in features]

@tool
def earthquake_adapter(lat: float, lon: float, tz_name: str = "UTC", radius_km: int = 200, days: int = 30, min_mag: float = 2.5) -> dict:
    """Get recent earthquake events near a location via USGS."""
    try:
        # Cache for 1 hour (3600 seconds)
        events = get_cached_or_live("usgs_earthquake", fetch_earthquake_live, 3600, lat, lon, radius_km, days, min_mag)
        return {
            "source": "usgs", 
            "timestamp": local_now_iso(tz_name), 
            "location": {"lat": lat, "lon": lon}, 
            "measurements": {"event_count": len(events), "events": events}
        }
    except Exception as e:
        return {"source": "usgs", "error": True, "raw_reference": str(e)}

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

TOOLS = [geocode, geo_adapter, weather_adapter, air_quality_adapter, fire_adapter, earthquake_adapter]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

llm_with_tools = ChatGroq(model="openai/gpt-oss-20b", temperature=0).bind_tools(TOOLS)

PLANNER_SYSTEM_PROMPT = (
    "You are TerraWatch's investigation planner. Given a location and a user question, "
    "decide which tools are relevant and call them to gather evidence.\n\n"
    "Rules:\n"
    "- Only call tools that are relevant to the question (e.g. don't call fire_adapter for an earthquake question).\n"
    "- Always pass the given lat/lon/tz_name to each tool — do not guess coordinates.\n"
    "- You may call multiple tools in one turn if several sources are relevant.\n"
    "- Once you have enough evidence to answer, stop calling tools and reply with a short text summary "
    "instead of a tool call.\n"
)

def planning_system(state: InvestigationState) -> InvestigationState:
    """Investigation planner + tool/source selector. Loops until the model stops requesting tools
    or max_iterations is hit."""
    if not state["messages"]:
        state["messages"] = [
            SystemMessage(content=PLANNER_SYSTEM_PROMPT),
            HumanMessage(content=(
                f"Location: {state['location']}\n"
                f"Question: {state['question']}"
            )),
        ]

    while state["iterations"] < state["max_iterations"]:
        state["iterations"] += 1
        ai_msg: AIMessage = llm_with_tools.invoke(state["messages"])
        state["messages"].append(ai_msg)

        if not ai_msg.tool_calls:
            break

        for call in ai_msg.tool_calls:
            tool_fn = TOOLS_BY_NAME.get(call["name"])
            if tool_fn is None:
                result = {"error": True, "raw_reference": f"Unknown tool: {call['name']}"}
            else:
                try:
                    result = tool_fn.invoke(call["args"])
                except Exception as e:
                    result = {"source": call["name"], "error": True, "raw_reference": str(e)}

            state["evidence"].append(result)
            state["messages"].append(
                ToolMessage(content=json.dumps(result), tool_call_id=call["id"])
            )

    return state

In [ ]:
class AnalysisResult(BaseModel):
    key_findings: List[str] = Field(description="Short factual observations drawn directly from the evidence.")
    risk_indicators: List[str] = Field(description="Qualitative risk signals, e.g. 'elevated PM2.5 near active fire'.")
    missing_evidence: List[str] = Field(default_factory=list, description="Sources/data still needed to answer confidently.")
    needs_more_evidence: bool = Field(description="True if the planner should gather more evidence before finalizing.")
    confidence: float = Field(description="0.0-1.0 confidence in the findings given current evidence.")

analysis_parser = JsonOutputParser(pydantic_object=AnalysisResult)

ANALYSIS_PROMPT = PromptTemplate(
    template=(
        "You are TerraWatch's analysis reasoner. You are given normalized evidence collected from "
        "environmental data APIs. Reason over the structured evidence only — do NOT invent numbers, "
        "distances, or correlations not present in the data.\n\n"
        "Question: {question}\n\n"
        "Evidence:\n{evidence}\n\n"
        "{format_instructions}\n"
    ),
    input_variables=["question", "evidence"],
    partial_variables={"format_instructions": analysis_parser.get_format_instructions()},
)

analysis_chain = ANALYSIS_PROMPT | llm | analysis_parser

def analsys_system(state: InvestigationState) -> AnalysisResult:
    """Analyze results / confidence assessment step. Flags whether more evidence is needed —
    the caller can feed that back into run_orchestrator for another loop."""
    evidence_str = json.dumps(state["evidence"], indent=2, default=str)
    raw = analysis_chain.invoke({"question": state["question"], "evidence": evidence_str})
    return AnalysisResult(**raw)